# VOI headline — scenario × β surface (X-01, VOI-01)

This notebook is the **M3 value-of-information headline** for the blog post framed in
[X-01](https://github.com/): *what is knowing realised age composition worth, in profit?*
We report **VOI-01** metrics — percentage improvement vs the P0 (age-blind) baseline **and**
absolute dollar deltas — on a **scenario × spoilage-sensitivity β** grid ([X-06](https://github.com/)).

**Board locks:** VOI-01=C (% headline + $ support) · VOI-03=B (paired bootstrap CI on $) ·
VOI-04=B (fine β grid, includes β=1) · SIM-02=C (shared CRN across scenarios).

> **Do not cite smoke-budget numbers as headline VOI.** ADR 0095 splits CI smoke presets from
> production defaults; only the production cell below is intended for citeable figures.

## Setup

Install the package in editable mode with notebook + viz extras, then build the Rust kernel
if it is not already importable:

```bash
uv sync --extra notebooks --extra viz --extra rust --python 3.11
cd crates/voi_py && uv run maturin develop --release
```

VOI CRN cells require **`BLUEBERRIES_VOI_BACKEND=rust`** and `blueberries_voi._core` (T-121 Wave F).
Without the extension, `run_voi_crn_cell` / `run_voi_sweep` will raise.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

# Rust kernel is required for VOI CRN cells (default backend per ADR 0127).
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")

import matplotlib.pyplot as plt
import numpy as np

from blueberries_voi.backend import rust_available, warn_fallback_once
from blueberries_voi.sim.alpha_tune import DEFAULT_TUNED_ALPHA_PATH
from blueberries_voi.sim.profit import DEFAULT_PROFIT_COSTS, ProfitCosts
from blueberries_voi.voi import (
    PRODUCTION_BETAS,
    PRODUCTION_N_BURN,
    PRODUCTION_ROLLOUT_H,
    SMOKE_BETAS,
    VOI_SCENARIOS,
    VoISweepResult,
    assert_beta_one_voi_near_zero,
    paired_bootstrap_ci,
    run_voi_sweep,
    voi_vs_p0,
)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").exists():
    REPO_ROOT = REPO_ROOT.parent
FIG_M3 = REPO_ROOT / "figures" / "m3"
FIG_M3.mkdir(parents=True, exist_ok=True)

warn_fallback_once()
if not rust_available():
    raise RuntimeError(
        "blueberries_voi._core is not importable. Build with:\n"
        "  cd crates/voi_py && uv run maturin develop --release"
    )

print(f"Backend: {os.environ['BLUEBERRIES_VOI_BACKEND']!r}  rust_available={rust_available()}")
print(f"Figure export target: {FIG_M3}")

Backend: 'rust'  rust_available=True
Figure export target: /home/oliver/blog/blueberries-voi/figures/m3


## CAL-01 base case (MWF + FreshNet)

Under **[CAL-01](.team/plans/CAL-01-calendar-realism.md)** the scientific base case is no longer
daily delivery with i.i.d. demand. ADR **0112** locks **Mon / Wed / Fri** deliveries (LT=1) with
day-indexed protection; ADR **0113** / **0115** supply **FreshNet-derived** calendar DOW×week demand.

Implications for this notebook:

- Physics still ticks daily; orders are gated to Sun/Tue/Thu placement → Mon/Wed/Fri delivery.
- Production burn-in (`PRODUCTION_N_BURN=28`) and rollout horizon (`PRODUCTION_ROLLOUT_H`) follow
  **weekly multiples of 7** under periodic MWF age (T-083).
- **Prior citeable VOI numbers from the daily/i.i.d. base case are invalidated** — regenerate after
  CAL-01 lands on `main` (see `.team/changelog.md`).

## Scaffold economics disclaimer (`DEFAULT_PROFIT_COSTS`)

Episode profit uses **SIM-01=B** margin − waste − stockout via `sim.profit`. When callers omit
`costs`, production paths resolve to the shared scaffold:

| Field | Scaffold value |
| --- | ---: |
| `unit_margin` | 2.0 |
| `waste_cost` | 1.5 |
| `stockout_penalty` | 3.0 |

These are **uncalibrated placeholder economics** (ADR 0104 / T-043). Dollar VOI deltas inherit this
scale; percentage VOI vs P0 is scale-invariant but still reflects scaffold *relative* costs.
Do not present absolute dollars as calibrated store P&L without refitting costs.

In [2]:
assert isinstance(DEFAULT_PROFIT_COSTS, ProfitCosts)
DEFAULT_PROFIT_COSTS

ProfitCosts(unit_margin=2.0, waste_cost=1.5, stockout_penalty=3.0)

## VOI-01 metric: `voi_vs_p0` (% and $)

`voi.metric.voi_vs_p0(profit_scenario, profit_p0)` returns a `VoIMetric` with:

- **`absolute_delta`** — scenario profit minus P0 profit ($)
- **`pct_vs_p0`** — `absolute_delta / profit_p0` (headline %; denominator is P0 per VOI-01)

Raises `ValueError` when P0 profit is exactly zero (unstable %).

In [3]:
example = voi_vs_p0(profit_scenario=110.0, profit_p0=100.0)
print(f"Δ$ = {example.absolute_delta:.1f}")
print(f"% vs P0 = {100.0 * example.pct_vs_p0:.1f}%")

at_beta_one = voi_vs_p0(50.0, 50.0)
assert at_beta_one.absolute_delta == 0.0 and at_beta_one.pct_vs_p0 == 0.0

Δ$ = 10.0
% vs P0 = 10.0%


## VOI-03 paired bootstrap: `paired_bootstrap_ci`

`voi.bootstrap.paired_bootstrap_ci` resamples **replication indices** of already-paired
per-rep profit differences (scenario − P0). It does **not** shuffle scenario/P0 labels
independently — preserving the CRN pairing from SIM-02.

The sweep stores CI bounds on **absolute $** deltas; percentage CIs are not bootstrapped separately.

In [4]:
rng = np.random.default_rng(0)
paired_deltas = [1.2, 0.8, 1.5, 0.9, 1.1]  # toy per-rep scenario − P0
ci = paired_bootstrap_ci(paired_deltas, n_bootstrap=500, alpha=0.05, rng=rng)
ci

BootstrapCI(mean=1.1, low=0.9, high=1.32, n_bootstrap=500, alpha=0.05)

## `run_voi_sweep` — smoke vs production budgets (ADR 0095)

`voi.sweep.run_voi_sweep` orchestrates the full **scenario × β** surface. Same API, two budget
regimes:

| Knob | Smoke (`smoke=True`) | Production (`smoke=False`) |
| --- | --- | --- |
| β grid | `SMOKE_BETAS` = (1.0, 2.0) | `PRODUCTION_BETAS` — linspace(1, 4, 10) |
| `n_burn` | 1 | 28 (4 weeks, MWF-aligned) |
| `n_score` | 2 | 60 |
| `n_replications` | 2 | 20 |
| `n_bootstrap` | 32 | 200 |
| `filter_n` | 16 | 64 |
| `H` (rollout) | 2 | `PRODUCTION_ROLLOUT_H` (7× preset) |
| `n_rollout_paths` | 1 | 8 |
| α table | omitted → fixed α=0.9 + `smoke_cool_shipments()` | **`alpha_table_path` required** (CTL-03) |

CI pytest uses smoke only. Headline blog figures must use **production** budgets + tuned α artifact.

In [5]:
# Values mirror voi/sweep.py ADR 0095 presets (see module docstring).
SMOKE_BUDGETS = {
    "n_burn": 1,
    "n_score": 2,
    "n_replications": 2,
    "n_bootstrap": 32,
    "filter_n": 16,
    "H": 2,
    "n_rollout_paths": 1,
}
PROD_BUDGETS = {
    "n_burn": PRODUCTION_N_BURN,
    "n_score": 60,
    "n_replications": 20,
    "n_bootstrap": 200,
    "filter_n": 64,
    "H": PRODUCTION_ROLLOUT_H,
    "n_rollout_paths": 8,
}
budget_rows = [("betas", SMOKE_BETAS, PRODUCTION_BETAS)] + [
    (k, SMOKE_BUDGETS[k], PROD_BUDGETS[k]) for k in SMOKE_BUDGETS
]
print(f"{'knob':<18} {'smoke':>24} {'production':>24}")
print("-" * 68)
for name, smoke_val, prod_val in budget_rows:
    print(f"{name:<18} {str(smoke_val):>24} {str(prod_val):>24}")
print(f"\nVOI scenario columns: {VOI_SCENARIOS}")

knob                                  smoke               production
--------------------------------------------------------------------
betas                            (1.0, 2.0) (1.0, 1.3333333333333333, 1.6666666666666665, 2.0, 2.333333333333333, 2.6666666666666665, 3.0, 3.333333333333333, 3.6666666666666665, 4.0)
n_burn                                    1                       28
n_score                                   2                       60
n_replications                            2                       20
n_bootstrap                              32                      200
filter_n                                 16                       64
H                                         2                       28
n_rollout_paths                           1                        8

VOI scenario columns: ('P0', 'P1', 'F1', 'F1s', 'F2a', 'F2', 'B-state')


## Interactive smoke replication (REDUCED)

The cell below runs a **deliberately tiny** sweep for notebook interactivity. It exercises the same
code paths as CI smoke but with `n_replications=1` and a three-scenario slice (P0, P1, B-state).

**Not citeable.** Use for wiring checks, plot scaffolding, and β=1 sanity only.

In [6]:
SMOKE_SCENARIOS = ["P0", "P1", "B-state"]
ROOT_SEED = 42

smoke_result = run_voi_sweep(
    smoke=True,
    root_seed=ROOT_SEED,
    scenarios=SMOKE_SCENARIOS,
    # Extra reductions for interactive notebook turnaround:
    n_replications=1,
    n_bootstrap=16,
    n_burn=1,
    n_score=2,
)
assert smoke_result.smoke is True
plot_source = smoke_result  # default for plots/summary; production cell may replace
smoke_result.to_jsonable()["arms"][:4]

[{'scenario': 'P1',
  'beta': 1.0,
  'absolute_delta': 0.0,
  'pct_vs_p0': 0.0,
  'ci_low': 0.0,
  'ci_high': 0.0,
  'n_replications': 1},
 {'scenario': 'B-state',
  'beta': 1.0,
  'absolute_delta': 1.5,
  'pct_vs_p0': 0.06382978723404255,
  'ci_low': 1.5,
  'ci_high': 1.5,
  'n_replications': 1},
 {'scenario': 'P1',
  'beta': 2.0,
  'absolute_delta': 1.5,
  'pct_vs_p0': 0.06382978723404255,
  'ci_low': 1.5,
  'ci_high': 1.5,
  'n_replications': 1},
 {'scenario': 'B-state',
  'beta': 2.0,
  'absolute_delta': 1.5,
  'pct_vs_p0': 0.06382978723404255,
  'ci_low': 1.5,
  'ci_high': 1.5,
  'n_replications': 1}]

## β = 1 sanity check (ENG-04)

At **β = 1** (reference spoilage sensitivity), knowledge scenarios should not beat P0 by much under
shared physics — VOI ≈ 0. `assert_beta_one_voi_near_zero` gates this on **absolute $** with a
generous tolerance under smoke horizons (`tol=50` by default).

In [7]:
beta_one_arms = [a for a in smoke_result.arms if abs(a.beta - 1.0) < 1e-12]
for arm in beta_one_arms:
    print(
        f"{arm.scenario:8s} β=1  Δ$={arm.absolute_delta:8.3f}  "
        f"%={100 * arm.pct_vs_p0:6.2f}%  CI$=[{arm.ci_low:.3f}, {arm.ci_high:.3f}]"
    )

assert_beta_one_voi_near_zero(smoke_result, tol=50.0)
print("β=1 gate passed (smoke tolerance).")

P1       β=1  Δ$=   0.000  %=  0.00%  CI$=[0.000, 0.000]
B-state  β=1  Δ$=   1.500  %=  6.38%  CI$=[1.500, 1.500]
β=1 gate passed (smoke tolerance).


## Production sweep (citeable numbers)

Set `RUN_PRODUCTION = True` to execute the full VOI-04 grid. Expect **long wall-clock** (fine β ×
all rungs × SW+rollout). Requirements:

1. Tuned CTL-03 α table at `experiments/tuned_alpha.json` (or override `ALPHA_TABLE`).
2. CAL-01 base case already baked into the Rust kernel.
3. Production budgets from `run_voi_sweep(smoke=False, ...)` defaults.

Leave `RUN_PRODUCTION = False` during exploratory editing; commit figures only from a completed
production run logged with `root_seed` and artifact paths.

In [12]:
RUN_PRODUCTION = True
ALPHA_TABLE = REPO_ROOT / DEFAULT_TUNED_ALPHA_PATH

production_result: VoISweepResult | None = None

if RUN_PRODUCTION:
    if not ALPHA_TABLE.is_file():
        raise FileNotFoundError(
            f"Production VOI requires tuned α table at {ALPHA_TABLE}. "
            "Run alpha_tune / M2 ladder first (T-029)."
        )
    production_result = run_voi_sweep(
        smoke=False,
        root_seed=ROOT_SEED,
        alpha_table_path=ALPHA_TABLE,
        # Defaults: PRODUCTION_BETAS, n_burn=28, n_score=60, n_rep=20, ...
    )
    assert production_result.smoke is False
    assert len(production_result.betas) >= 10
    assert 1.0 in production_result.betas
    print("Production sweep complete — numbers are citeable subject to cost calibration disclaimer.")
else:
    print(
        "Production sweep skipped (RUN_PRODUCTION=False). "
        "Plots below use smoke_result for layout; re-run with production for headline figures."
    )

plot_source = production_result if production_result is not None else smoke_result
plot_source.smoke

FileNotFoundError: Production VOI requires tuned α table at /home/oliver/blog/blueberries-voi/experiments/tuned_alpha.json. Run alpha_tune / M2 ladder first (T-029).

## Figures → `figures/m3/`

Static matplotlib outputs for the blog (ENG-03=A). The library keeps matplotlib out of `voi/`;
plotting lives here (or in `viz.voi` for the simple β-curve hook).

- **Heatmap** — % VOI vs P0 (scenario × β)
- **Ribbon** — absolute $ VOI vs β with paired-bootstrap CI bands per scenario

In [9]:
def _arm_grid(result: VoISweepResult) -> tuple[list[str], list[float], dict[tuple[str, float], object]]:
    scenarios = [s for s in result.scenarios if s != "P0"]
    betas = list(result.betas)
    lookup = {(a.scenario, float(a.beta)): a for a in result.arms}
    return scenarios, betas, lookup


def plot_voi_pct_heatmap(result: VoISweepResult, *, out_path: Path) -> Path:
    scenarios, betas, lookup = _arm_grid(result)
    mat = np.full((len(scenarios), len(betas)), np.nan)
    for i, scen in enumerate(scenarios):
        for j, beta in enumerate(betas):
            arm = lookup[(scen, float(beta))]
            mat[i, j] = 100.0 * float(arm.pct_vs_p0)

    fig, ax = plt.subplots(figsize=(8.0, 0.55 * len(scenarios) + 2.0))
    im = ax.imshow(mat, aspect="auto", origin="lower", cmap="RdYlGn")
    ax.set_xticks(range(len(betas)), labels=[f"{b:.2g}" for b in betas])
    ax.set_yticks(range(len(scenarios)), labels=scenarios)
    ax.set_xlabel("β (spoilage sensitivity)")
    ax.set_ylabel("Knowledge scenario")
    tag = "smoke" if result.smoke else "production"
    ax.set_title(f"VOI % vs P0 — scenario × β ({tag})")
    for i in range(len(scenarios)):
        for j in range(len(betas)):
            if np.isfinite(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:.1f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="% vs P0", shrink=0.85)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path


def plot_voi_dollar_ribbons(result: VoISweepResult, *, out_path: Path) -> Path:
    scenarios, betas, lookup = _arm_grid(result)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(scenarios), 1)))

    for scen, color in zip(scenarios, colors, strict=True):
        xs, ys, lo, hi = [], [], [], []
        for beta in betas:
            arm = lookup[(scen, float(beta))]
            xs.append(float(beta))
            ys.append(float(arm.absolute_delta))
            lo.append(float(arm.ci_low))
            hi.append(float(arm.ci_high))
        ax.plot(xs, ys, marker="o", label=scen, color=color)
        ax.fill_between(xs, lo, hi, alpha=0.2, color=color, linewidth=0)

    ax.axhline(0.0, color="0.5", linewidth=0.8)
    ax.set_xlabel("β")
    ax.set_ylabel("Absolute Δ$ vs P0")
    tag = "smoke" if result.smoke else "production"
    ax.set_title(f"VOI vs β with paired-bootstrap CI ({tag})")
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

In [10]:
# Resolve plot_source if figures are run without the production cell.
_production = globals().get("production_result")
plot_source = _production if _production is not None else smoke_result

tag = "smoke" if plot_source.smoke else "production"
heatmap_path = FIG_M3 / f"voi_pct_heatmap_{tag}.png"
ribbon_path = FIG_M3 / f"voi_dollar_ribbon_{tag}.png"

plot_voi_pct_heatmap(plot_source, out_path=heatmap_path)
plot_voi_dollar_ribbons(plot_source, out_path=ribbon_path)

print(f"Wrote heatmap: {heatmap_path}")
print(f"Wrote ribbon:   {ribbon_path}")

# Optional: also use the library hook for a simple % curve (no CI ribbons).
from blueberries_voi.viz.voi import plot_voi_vs_beta

curve_path = FIG_M3 / f"voi_vs_beta_pct_{tag}.png"
plot_voi_vs_beta(plot_source, out_path=curve_path, use_pct=True)
print(f"Wrote curve:    {curve_path}")

Wrote heatmap: /home/oliver/blog/blueberries-voi/figures/m3/voi_pct_heatmap_smoke.png
Wrote ribbon:   /home/oliver/blog/blueberries-voi/figures/m3/voi_dollar_ribbon_smoke.png
Wrote curve:    /home/oliver/blog/blueberries-voi/figures/m3/voi_vs_beta_pct_smoke.png


## Summary table (current `plot_source`)

Headline **% vs P0** per arm. Dollar CIs are on the paired absolute deltas (see ribbon plot).

In [11]:
# Resolve plot_source if this cell is run without the production cell (e.g. Run Selected).
_production = globals().get("production_result")
plot_source = _production if _production is not None else smoke_result

print(f"{'scenario':10s} {'beta':>5s} {'Δ$':>10s} {'% vs P0':>10s} {'CI low $':>10s} {'CI high $':>10s}")
print("-" * 62)
for arm in sorted(plot_source.arms, key=lambda a: (a.scenario, a.beta)):
    print(
        f"{arm.scenario:10s} {arm.beta:5.2f} {arm.absolute_delta:10.3f} "
        f"{100 * arm.pct_vs_p0:9.2f}% {arm.ci_low:10.3f} {arm.ci_high:10.3f}"
    )

scenario    beta         Δ$    % vs P0   CI low $  CI high $
--------------------------------------------------------------
B-state     1.00      1.500      6.38%      1.500      1.500
B-state     2.00      1.500      6.38%      1.500      1.500
P1          1.00      0.000      0.00%      0.000      0.000
P1          2.00      1.500      6.38%      1.500      1.500
